# 04. Presentation Visuals Generator
Generates publication-quality figures for the LaTeX Beamer presentation.
Saves outputs directly to the `presentation/` folder.

In [1]:
# 1. Environment Setup & Imports
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.manifold import TSNE
import ast 

# --- PATH CONFIGURATION ---
# Detect Project Root (assuming notebook is in ./notebooks/)
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'notebooks':
    PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, '..'))
else:
    PROJECT_ROOT = os.path.abspath(current_dir)

# Add 'src' to Python Path
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Define Key Paths based on your structure
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "glioma_dataset.h5")
SPLIT_FILE = os.path.join(PROJECT_ROOT, "data", "splits", "strict_splits.json")
BASELINE_CSV_PATH = os.path.join(PROJECT_ROOT, "results", "baseline_metrics.csv")

# Specific Experiment Weights
EXP_FOLDER = "CVAE_PositionAware_20260121_0936"
WEIGHTS_PATH = os.path.join(PROJECT_ROOT, "experiments", EXP_FOLDER, "best_weights.weights.h5")

# Output Directory (Directly to presentation folder)
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "presentation")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📂 Project Root: {PROJECT_ROOT}")
print(f"📂 Reading Weights from: {WEIGHTS_PATH}")
print(f"📂 Saving Figures to:    {OUTPUT_DIR}")

# Import Project Modules
from src.data.loader import GliomaDataLoader
from src.utils.config import ExperimentConfig
from src.models.cvae import CVAE

2026-02-04 10:09:12.495266: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-04 10:09:27.355756: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


📂 Project Root: /home/bv/Documents/job/incdsb/glioma-synthetic-data
📂 Reading Weights from: /home/bv/Documents/job/incdsb/glioma-synthetic-data/experiments/CVAE_PositionAware_20260121_0936/best_weights.weights.h5
📂 Saving Figures to:    /home/bv/Documents/job/incdsb/glioma-synthetic-data/presentation


In [2]:
# 2. Configuration & Model Loading
# We recreate the exact config used in training
config = ExperimentConfig(
    experiment_name="Visuals_Gen",
    input_shape=[1738, 1],
    num_classes=7,
    latent_dim=16,
    batch_size=128,
    epochs=50, 
    learning_rate=0.0001,
    encoder={
        "filters": [32, 64, 128], "kernels": [7, 7, 5], "strides": [2, 2, 2],
        "use_skip_gap": False, "use_global_pooling": False
    },
    decoder={
        "initial_filters": 128, "filters": [64, 32, 16], "kernels": [5, 7, 7], 
        "strides": [2, 2, 2], "use_dense_projection": True
    },
    loss_weights={"beta": 0.5, "alpha": 2.0}
)

# Initialize & Load
try:
    model = CVAE(config)
    # Build graph with dummy input to initialize weights
    model.build(input_shape=[(None, 1738, 1), (None, 7)])
    model.load_weights(WEIGHTS_PATH)
    print("✅ CVAE Model loaded successfully.")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("Check if the weights path is correct.")

2026-02-04 10:09:35.133287: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


✅ CVAE Model loaded successfully.


In [ ]:
# 3. Load Validation Data (Memory-Mapped + Stratified Sampling for RAM Efficiency)
try:
    loader = GliomaDataLoader(DATA_PATH, split_file=SPLIT_FILE, memory_mapped=True)
    # GliomaDataLoader is iterable - get first fold
    fold_data = next(iter(loader))  # Fold 0
    X_val_full, y_val_full = fold_data['val']
    
    # Stratified sampling: 50 samples per class (representative across all 7 classes)
    n_per_class = 50
    X_val_list = []
    y_val_class_list = []
    
    for class_id in range(7):
        class_indices = np.where(y_val_full == class_id)[0]
        if len(class_indices) > 0:
            # Sample min(n_per_class, available)
            sample_size = min(n_per_class, len(class_indices))
            sampled_idx = np.random.choice(class_indices, sample_size, replace=False)
            X_val_list.append(X_val_full[sampled_idx])
            y_val_class_list.append(y_val_full[sampled_idx])
    
    X_val = np.concatenate(X_val_list, axis=0)
    y_val_class = np.concatenate(y_val_class_list, axis=0)
    
    # Preprocessing
    X_val_exp = np.expand_dims(X_val, axis=-1)
    y_val_oh = tf.keras.utils.to_categorical(y_val_class, num_classes=7)
    
    print(f"✅ Loaded Stratified Validation Sample: {X_val_exp.shape}")
    print(f"   Class distribution: {np.bincount(y_val_class)}")
    
    # Keep reference to full data for fallback
    X_val_full_ref = X_val_full
    y_val_full_ref = y_val_full
    
except Exception as e:
    print(f"❌ Error loading data: {e}")

Loading data from /home/bv/Documents/job/incdsb/glioma-synthetic-data/data/processed/glioma_dataset.h5...
Data Loaded. Shape: (294710, 1738)
Unique Samples: 58
Sample ID Example: 'HF-305_v4b_1_1.h5_6' (Type: <class 'numpy.str_'>)
Loading frozen splits from /home/bv/Documents/job/incdsb/glioma-synthetic-data/data/splits/manual_splits.json
DEBUG: Split asks for 'HF-2544_V1B_1.h5_3'
DEBUG: Loader has 'HF-305_v4b_1_1.h5_6'
✅ Loaded Stratified Validation Sample: (300, 1738, 1)
   Class distribution: [ 0 50 50 50 50 50 50]


In [ ]:
# --- NEW CELL: Figure 1 - The Biological Challenge (Signal vs. Noise) ---
print("Generating Figure 1: Median Spectra with Biological Peaks...")

# Assuming indices map roughly to 400-1800 cm^-1 Raman shift. 
# Adjust these indices based on your specific spectrometer calibration!
peak_indices = {
    "Phenylalanine\n(~1003 cm⁻¹)": 600,  
    "Lipids/CH2\n(~1440 cm⁻¹)": 1040,
    "Amide I\n(~1650 cm⁻¹)": 1250
}

plt.figure(figsize=(12, 6))
colors = plt.cm.tab10(np.linspace(0, 1, 7))
class_names = ['Normal', 'LGm-1', 'LGm-2', 'LGm-3', 'LGm-4', 'LGm-5', 'LGm-6']
x_axis = np.arange(1738)

for c in range(7):
    class_mask = np.where(y_val_class == c)[0]
    if len(class_mask) > 0:
        class_spectra = X_val[class_mask]
        mean_spec = np.mean(class_spectra, axis=0)
        std_spec = np.std(class_spectra, axis=0)
        
        # Plot mean line
        plt.plot(x_axis, mean_spec, color=colors[c], label=class_names[c], linewidth=1.5)
        # Plot 1-std deviation shading (highly transparent to show overlap)
        plt.fill_between(x_axis, mean_spec - std_spec, mean_spec + std_spec, 
                         color=colors[c], alpha=0.1)

# Add vertical dashed lines for biological peaks
for peak_name, idx in peak_indices.items():
    plt.axvline(x=idx, color='gray', linestyle='--', alpha=0.7)
    plt.text(idx + 10, plt.ylim()[1] * 0.9, peak_name, color='black', fontsize=9, 
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

plt.title("Raman Spectra: The Generalization Challenge\n(Overlapping Variance Obscures Subtle Biological Peaks)", fontsize=14, fontweight='bold')
plt.xlabel("Wavelength Index (Raman Shift)", fontsize=12)
plt.ylabel("Normalized Intensity", fontsize=12)
plt.xlim(0, 1738)
plt.legend(loc='upper right', ncol=2)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, "Fig1_Biological_Challenge.pdf")
plt.savefig(save_path, dpi=300)
plt.show()

In [ ]:
# --- NEW CELL: Figure 2 - The Information Bottleneck (Feature t-SNE) ---
print("Generating Figure 2: Real vs. Synthetic Feature Space t-SNE...")

# 1. Generate perfectly balanced Synthetic Data
n_synth_per_class = 50
z_synth = np.random.normal(0, 1, size=(7 * n_synth_per_class, 16))
y_synth_classes = np.repeat(np.arange(7), n_synth_per_class)
y_synth_oh = tf.keras.utils.to_categorical(y_synth_classes, num_classes=7)

synth_spectra = model.decoder.predict([z_synth, y_synth_oh], verbose=0)
synth_spectra = np.squeeze(synth_spectra, axis=-1)

# 2. Run t-SNE on both spaces separately to avoid projection warping
print("Running t-SNE on Real Data...")
tsne_real = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_val)

print("Running t-SNE on Synthetic Data...")
tsne_synth = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(synth_spectra)

# 3. Plot Side-by-Side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
colors = plt.cm.tab10(np.linspace(0, 1, 7))

# Plot Real
scatter1 = ax1.scatter(tsne_real[:, 0], tsne_real[:, 1], c=y_val_class, cmap='tab10', alpha=0.7, edgecolors='k')
ax1.set_title("A. Real Validation Spectra (TRTR)\nDominated by batch effects and noise", fontsize=12, fontweight='bold')
ax1.set_xticks([]); ax1.set_yticks([])

# Plot Synthetic
scatter2 = ax2.scatter(tsne_synth[:, 0], tsne_synth[:, 1], c=y_synth_classes, cmap='tab10', alpha=0.7, edgecolors='k')
ax2.set_title("B. $\\beta$-CVAE Denoised Spectra (TSTR)\nIsolated biological sub-grade manifolds", fontsize=12, fontweight='bold')
ax2.set_xticks([]); ax2.set_yticks([])

# Add unified colorbar
cbar = fig.colorbar(scatter2, ax=[ax1, ax2], ticks=range(7), pad=0.02)
cbar.ax.set_yticklabels(class_names)
cbar.set_label("Glioma Sub-Grade", rotation=270, labelpad=20)

save_path = os.path.join(OUTPUT_DIR, "Fig2_Information_Bottleneck_tSNE.pdf")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- NEW CELL: Figure 3 - Denoising Validation (Zoomed Overlay) ---
print("Generating Figure 3: Denoised Biological Peak...")

idx = np.random.choice(np.where(y_val_class == 5)[0]) # Pick a GBM sample
real_spec = X_val_exp[idx:idx+1]
label_spec = y_val_oh[idx:idx+1]
recon_spec, _, _, _ = model([real_spec, label_spec])

plt.figure(figsize=(10, 5))

# Plot full spectra in background
plt.plot(real_spec[0, :, 0], color='gray', alpha=0.5, label='Raw Input (Noisy)', linewidth=1)
plt.plot(recon_spec[0, :, 0], color='#D32F2F', label='$\\beta$-CVAE Reconstruction', linewidth=2)

# Zoom in on a specific noisy peak area (e.g., indices 500 to 800)
# Adjust these bounds to highlight the best looking denoised peak in your specific data!
zoom_start, zoom_end = 500, 800 

axins = plt.axes([0.45, 0.4, 0.4, 0.4]) # [left, bottom, width, height]
axins.plot(real_spec[0, :, 0], color='gray', alpha=0.7)
axins.plot(recon_spec[0, :, 0], color='#D32F2F', linewidth=2)
axins.set_xlim(zoom_start, zoom_end)
axins.set_ylim(np.min(recon_spec[0, zoom_start:zoom_end, 0]) * 0.8, 
               np.max(recon_spec[0, zoom_start:zoom_end, 0]) * 1.2)
axins.set_title("Zoom: High-Frequency Noise Removal", fontsize=10)

plt.title("Classification by Reconstruction (CbR): Pure Physics Mapping", fontsize=14, fontweight='bold')
plt.legend(loc='upper right')

save_path = os.path.join(OUTPUT_DIR, "Fig3_Denoised_Peak.pdf")
plt.savefig(save_path, dpi=300)
plt.show()

In [ ]:
# --- NEW CELL: Figure 4 - SHAP Global Importance & RGB Mapping Recipe ---
print("Generating Figure 4: SHAP Feature Importance...")
import xgboost as xgb
import shap

# 1. Load your best TSTR XGBoost model from the overnight run
# (Update fold_0 to whichever fold performed best!)
MODEL_PATH = os.path.join(PROJECT_ROOT, "results", "models", "tstr", "xgb_fold_0.json")

try:
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(MODEL_PATH)
    
    # 2. Compute SHAP values on the validation set
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_val) # X_val is (350, 1738)
    
    # 3. Plot Global Feature Importance (Top 10 Wavelengths)
    plt.figure(figsize=(8, 6))
    shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=10, show=False)
    plt.title("XGBoost TSTR: Top Biochemical Signatures (SHAP)", fontweight='bold')
    
    save_path = os.path.join(OUTPUT_DIR, "Fig4_SHAP_Importance.pdf")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # --- RGB MAPPING RECIPE ---
    # Find the absolute top 3 indices
    mean_abs_shap = np.abs(shap_values).mean(axis=(0, 1)) if len(np.shape(shap_values)) == 3 else np.abs(shap_values).mean(axis=0)
    top_3_indices = np.argsort(mean_abs_shap)[-3:][::-1]
    
    print(f"\n🧪 BIOMARKER RGB MAPPING FOUND!")
    print(f"Top 3 most important Raman shift indices: {top_3_indices}")
    print("\nTo generate the RGB Tumor Map on a raw slide, run:")
    print(f"  R_channel = slide_cube[:, :, {top_3_indices[0]}]")
    print(f"  G_channel = slide_cube[:, :, {top_3_indices[1]}]")
    print(f"  B_channel = slide_cube[:, :, {top_3_indices[2]}]")
    print("  RGB_Image = np.dstack([R_channel, G_channel, B_channel])")
    
except Exception as e:
    print(f"❌ Could not run SHAP. Ensure xgboost and shap are installed and the model path is correct. Error: {e}")

In [ ]:
# --- NEW CELL: Figure 4B - SHAP Spectral Overlay (Interpretable Biology) ---
print("Generating Figure 4B: SHAP Spectral Overlay...")
import matplotlib.cm as cm
from matplotlib.collections import LineCollection

try:
    # 1. Aggregate SHAP values to get a single importance vector
    # Handle both list output (older SHAP versions) and 3D array output
    if isinstance(shap_values, list):
        mean_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
    elif len(np.shape(shap_values)) == 3:
        mean_shap = np.abs(shap_values).mean(axis=(0, 2))
    else:
        mean_shap = np.abs(shap_values).mean(axis=0)

    # Normalize SHAP values to [0, 1] for coloring/alpha
    shap_norm = mean_shap / mean_shap.max()

    # 2. Get the global mean spectrum to act as the visual base
    mean_spectrum = np.mean(X_val, axis=0)
    x_axis = np.arange(1738)

    plt.figure(figsize=(14, 6))
    ax = plt.gca()

    # 3. Plot the background SHAP intensity map
    # We use vertical spans where the alpha (transparency) and color map to SHAP importance
    cmap = plt.cm.get_cmap('Reds')
    
    # To render efficiently, we only shade regions with at least 5% relative importance
    for i in range(len(x_axis)):
        if shap_norm[i] > 0.05:  
            # Color intensity and opacity scale with SHAP importance
            ax.axvspan(i - 0.5, i + 0.5, color=cmap(shap_norm[i]), 
                       alpha=shap_norm[i] * 0.7, zorder=1, lw=0)

    # 4. Plot the mean spectrum over the heatmap
    ax.plot(x_axis, mean_spectrum, color='black', linewidth=1.5, zorder=2, label='Mean Validation Spectrum')

    # 5. Highlight and label the absolute Top 5 most important frequencies
    top_5_indices = np.argsort(mean_shap)[-5:][::-1]
    
    for rank, idx in enumerate(top_5_indices):
        ax.axvline(x=idx, color='darkred', linestyle='--', linewidth=1.5, zorder=3)
        # Annotate the peak. (If you know the actual Raman shifts in cm^-1 for these indices, 
        # you can map them here instead of just printing the raw index!)
        ax.annotate(f'#{rank+1}\nIdx: {idx}', 
                    xy=(idx, mean_spectrum[idx]), 
                    xytext=(idx + 20, mean_spectrum[idx] + 0.15),
                    arrowprops=dict(facecolor='darkred', arrowstyle='->', alpha=0.7),
                    fontsize=10, fontweight='bold', color='darkred',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1))

    plt.title("Biochemical Interpretability: XGBoost Feature Importance Mapped to Spectral Morphology", 
              fontsize=14, fontweight='bold')
    plt.xlabel("Wavelength Index (Raman Shift)", fontsize=12)
    plt.ylabel("Normalized Intensity", fontsize=12)
    plt.xlim(0, 1738)
    
    # Create a custom legend for the SHAP heatmap
    import matplotlib.patches as mpatches
    shap_patch = mpatches.Patch(color='red', alpha=0.5, label='SHAP Feature Importance')
    ax.legend(handles=[ax.get_legend_handles_labels()[0][0], shap_patch], loc='upper right')

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, "Fig4B_SHAP_Spectrum_Overlay.pdf")
    plt.savefig(save_path, dpi=300)
    plt.show()

    print(f"✅ Saved: {save_path}")

except Exception as e:
    print(f"❌ Could not generate SHAP Spectral Overlay. Error: {e}")

In [ ]:
# 6. Figure C: Reconstruction (Mirror Plot) - Direct Class Sampling
print("Generating Figure C...")

# Pick a GBM sample (Class 5) - sample directly without loading full dataset
target_class = 5
class_indices = np.where(y_val_class == target_class)[0]

if len(class_indices) > 0:
    # Randomly pick one representative sample from target class
    idx = np.random.choice(class_indices)
else:
    # Fallback: use first available sample
    idx = 0
    target_class = y_val_class[idx]

real_spec = X_val_exp[idx:idx+1]
label_spec = y_val_oh[idx:idx+1]
recon_spec, _, _, _ = model([real_spec, label_spec])

x_axis = np.arange(1738)

plt.figure(figsize=(10, 5))

# Top: Real
plt.subplot(2, 1, 1)
plt.plot(x_axis, real_spec[0, :, 0], color='black', linewidth=1.2, label='Real')
plt.fill_between(x_axis, real_spec[0, :, 0], color='black', alpha=0.05)
plt.xlim(0, 1738)
plt.xticks([])
plt.yticks([])
plt.legend(loc='upper right', frameon=False)
plt.title(f"Spectral Reconstruction (Class {target_class})")

# Bottom: Recon
plt.subplot(2, 1, 2)
plt.plot(x_axis, recon_spec[0, :, 0], color='#D32F2F', linewidth=1.2, label='CVAE')
plt.fill_between(x_axis, recon_spec[0, :, 0], color='#D32F2F', alpha=0.05)
plt.xlim(0, 1738)
plt.yticks([])
plt.gca().invert_yaxis() # Mirror effect
plt.legend(loc='lower right', frameon=False)

plt.subplots_adjust(hspace=0.0)
save_path = os.path.join(OUTPUT_DIR, "FigC_Reconstruction_Mirror.pdf")
plt.savefig(save_path, bbox_inches='tight')
plt.close()
print(f"✅ Saved: {save_path}")

Generating Figure C...
✅ Saved: /home/bv/Documents/job/incdsb/glioma-synthetic-data/presentation/FigC_Reconstruction_Mirror.pdf


In [7]:
# 7. Figure D1: Class Centroids (z=0, fixed one-hot) vs. Actual Mean Spectra with Envelopes
print("Generating Figure D1: Class Centroids with Envelopes...")

# First, encode ALL validation data to compute per-class statistics
print("Encoding all validation data...")
_, _, z_val, _ = model.encoder(X_val_exp)

# Compute per-class mean spectra and standard deviations
class_names = ['Normal', 'LGm-1', 'LGm-2', 'LGm-3', 'LGm-4', 'LGm-5', 'LGm-6']
mean_spectra = {}
std_spectra = {}
generated_spectra = {}

for c in range(7):
    class_indices = np.where(y_val_class == c)[0]
    if len(class_indices) > 0:
        # Actual mean and std from validation data
        class_data = X_val[class_indices]
        mean_spectra[c] = np.mean(class_data, axis=0)
        std_spectra[c] = np.std(class_data, axis=0)
        
        # Generated spectrum at z=0 with class one-hot
        z_center = np.zeros((1, 16))
        label_onehot = tf.keras.utils.to_categorical(c, 7)
        label_onehot_batch = np.expand_dims(label_onehot, axis=0)
        gen_spec = model.decoder([z_center, label_onehot_batch])
        generated_spectra[c] = gen_spec[0, :, 0].numpy()

# Plot: 7 subplots, one per class
fig, axes = plt.subplots(7, 1, figsize=(12, 12))
x_axis = np.arange(1738)

for c in range(1,7):
    ax = axes[c]
    
    # Actual mean ± std (gray envelope)
    ax.fill_between(x_axis, 
                     mean_spectra[c] - std_spectra[c], 
                     mean_spectra[c] + std_spectra[c],
                     color='gray', alpha=0.3, label='Actual ±1σ')
    ax.plot(x_axis, mean_spectra[c], color='black', linewidth=2, label='Actual Mean')
    
    # Generated at z=0 (red)
    ax.plot(x_axis, generated_spectra[c], color='#D32F2F', linewidth=2, linestyle='--', label='Generated (z=0)')
    
    ax.set_title(f"Class {c}: {class_names[c]}", fontsize=11, fontweight='bold')
    ax.set_ylabel("Intensity", fontsize=9)
    ax.set_xlim(0, 1738)
    ax.legend(loc='upper right', fontsize=9)
    if c < 6:
        ax.set_xticks([])

axes[-1].set_xlabel("Wavelength Index", fontsize=10)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, "FigD1_ClassCentroids_Envelopes.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Saved: {save_path}")
print("   Comparison: Actual mean spectrum vs. generated spectrum at z=0 for each class")


Generating Figure D1: Class Centroids with Envelopes...
Encoding all validation data...


✅ Saved: /home/bv/Documents/job/incdsb/glioma-synthetic-data/presentation/FigD1_ClassCentroids_Envelopes.png
   Comparison: Actual mean spectrum vs. generated spectrum at z=0 for each class


In [5]:
# 7.2. Figure D2: Intra-class Latent Traversal (Two Representative Samples)
print("Generating Figure D2: Intra-class Latent Traversal...")

# Pick GBM class (5) and find 2 samples with latent codes reasonably far apart
target_class = 5
class_indices = np.where(y_val_class == target_class)[0]

if len(class_indices) >= 2:
    # Get latent encodings for all samples in this class
    X_class = X_val_exp[class_indices]
    _, _, z_class, _ = model.encoder(X_class)
    z_class_np = z_class.numpy()
    
    # Find the two samples with largest pairwise distance
    from scipy.spatial.distance import pdist, squareform
    distances = squareform(pdist(z_class_np, metric='euclidean'))
    idx1, idx2 = np.unravel_index(np.argmax(distances), distances.shape)
    
    z_start = z_class_np[idx1:idx1+1]
    z_end = z_class_np[idx2:idx2+1]
    
    # Interpolate between them
    steps = 8
    alphas = np.linspace(0, 1, steps)
    z_path = np.array([z_start[0] * (1-a) + z_end[0] * a for a in alphas])
    
    # Fixed one-hot label for GBM
    label_gbm = tf.keras.utils.to_categorical(target_class, 7)
    label_gbm_batch = np.tile(np.expand_dims(label_gbm, axis=0), (steps, 1))
    
    # Decode the path
    generated_path = model.decoder([z_path, label_gbm_batch])
    
    # Waterfall plot
    plt.figure(figsize=(12, 7))
    offset = 0.6
    colors = plt.cm.viridis(np.linspace(0, 1, steps))
    
    for i in range(steps):
        signal = generated_path[i, :, 0].numpy() + (i * offset)
        plt.plot(signal, color=colors[i], linewidth=1.5, alpha=0.8)
        
        # Text labels at endpoints
        if i == 0:
            plt.text(1750, signal[-1], "Sample A", color=colors[i], va='center', fontweight='bold', fontsize=10)
        elif i == steps - 1:
            plt.text(1750, signal[-1], "Sample B", color=colors[i], va='center', fontweight='bold', fontsize=10)
    
    plt.title(f"Intra-class Latent Traversal (GBM / {class_names[target_class]})\nSmooth transition between two representative samples", 
              fontsize=12, fontweight='bold')
    plt.xlabel("Wavelength Index", fontsize=11)
    plt.ylabel("Relative Intensity (offset for visibility)", fontsize=11)
    plt.xlim(0, 1738)
    
    save_path = os.path.join(OUTPUT_DIR, "FigD2_IntraClass_Traversal.pdf")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")
    print(f"   Latent space distance between samples: {distances[idx1, idx2]:.3f}")
    print(f"   Shows smooth variation within GBM class latent manifold")
else:
    print("❌ Could not generate Figure D2 - insufficient samples in GBM class")

print("\n🎉 All figures generated successfully!")


Generating Figure D2: Intra-class Latent Traversal...
✅ Saved: /home/bv/Documents/job/incdsb/glioma-synthetic-data/presentation/FigD2_IntraClass_Traversal.pdf
   Latent space distance between samples: 10.341
   Shows smooth variation within GBM class latent manifold

🎉 All figures generated successfully!
